In [ ]:
!pip install nselib
!pip install plotnine

In [ ]:
import pandas as pd
from nselib import capital_market
import requests
import plotnine
from plotnine import ggplot, aes, geom_line, labs, theme_minimal, theme, element_text
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Retrieving Indices

this will be implemented without functional call

this step is where we call the list of symbols for our stocks

In [ ]:
#obtaining the nifty fifty list
data = capital_market.nifty50_equity_list()

#converting to a dataframe
df = pd.DataFrame(data)

#grabbing the stock symbols
symbols = df['Symbol'].tolist()

#displaying
symbols

# Functional Commands

## Data Retrieval

Given the stock symbols, we will use the API to retrieve data

And then we return the data as a dataframe once we save it as a CSV file

The title input, is the title of the CSV

In [ ]:
def hist_data(symbols, title, start, end):

  # Initialize an empty list to store DataFrames for each symbol
  dfs = {}

  for i in range(1, len(symbols)+1):

      url = f'https://eodhd.com/api/eod/{symbols[i-1]}.NSE?from={start}&to={end}&api_token=6647280cd70cd1.45605436&fmt=json' # with free API token, I can only do over the past year
      nifty_data = requests.get(url).json()

      #print(data)

      # Convert data to DataFrame
      dfs[f'df{i}'] = pd.DataFrame(nifty_data)

      # changing the date to the index column
      dfs[f'df{i}'] = dfs[f'df{i}'][['date', 'close']].set_index('date')

      # renaming the 'close' column to the name of the symbol
      dfs[f'df{i}'] = dfs[f'df{i}'].rename(columns={'close': symbols[i-1]})

      #print(dfs[f'df{i}'])



  # Join the data frames in the dictionary
  result_df = pd.DataFrame()

  for df_name, df in dfs.items():
      result_df = result_df.join(df, how='outer')  # 'outer' joins all columns


  #saving as CSV
  result_df.to_csv(title)

  print(f'Stock data has been saved to {title}')

  return result_df


## Obtaining Returns

The dataframe of data from the previous

In [ ]:
def returns(symbols, ret, title):

  #now to find returns using pct_change()

  for i in range(1, len(symbols)+1):

    ret[symbols[i-1]] = ret[symbols[i-1]].pct_change()

    # 1. Drop NaN values
    ret = ret.dropna(subset=[symbols[i-1]])

    # 2. Convert to percentages
    ret[symbols[i-1]] = ret[symbols[i-1]] * 100

    # 3. Drop anything above 100%
    ret = ret[ret[symbols[i-1]] <= 100]


  ret.to_csv(title)  # Set index=False to avoid writing row numbers

  print(f'Returns Data has been saved to {title}')

  return ret

## Plotting

In [ ]:
def returns_plot(ret):

  ret = ret.reset_index()
  ret['date'] = pd.to_datetime(ret['date'])

  # Reshape the DataFrame to long format
  long_data = pd.melt(ret, id_vars=['date'], var_name='Stock', value_name='Return')

  # Get unique stock names
  unique_stocks = long_data['Stock'].unique()

  # Number of stocks per plot
  #this number has to be changed on the basis of which market we are referring to
  stocks_per_plot = 5


  # Create plots for each subset of stocks
  for i in range(0, len(unique_stocks), stocks_per_plot):
      subset_stocks = unique_stocks[i:i + stocks_per_plot]
      subset_data = long_data[long_data['Stock'].isin(subset_stocks)]

      # Create the plot
      plot = (
          ggplot(subset_data, aes(x='date', y='Return', color='Stock')) +
          geom_line(size=1.5) +
          labs(title=f'Stock Returns Over Time (Stocks {i + 1} to {i + stocks_per_plot})', x='Date', y='Return') +
          theme_minimal(base_size=12) +
          theme(
              figure_size=(12, 8),  # Adjust the size of the graph
              legend_position='top',  # Relocate the legend
              axis_text_x=element_text(rotation=45, hjust=1)
          )
      )

      # Save the plot
      plot.save(f'stock_returns_plot_{i // stocks_per_plot + 1}.png')


  return None

# Main

Implementing all of the functions

In [ ]:
#getting the desired dates

# Get the current date
current_date = datetime.now()

# Calculate the date 20 years ago
date_20_years_ago = current_date - relativedelta(years=20)

#convert to string
start = date_20_years_ago.strftime('%Y-%m-%d')
end = current_date.strftime('%Y-%m-%d')


#gathering everything
data_df = hist_data(symbols, 'Nifty50_hist.csv', start, end)
returns_df = returns(symbols, data_df, 'Nifty50_returns')
returns_plot(returns_df)